# [Student Name] — Individual Model Notebook
**Branch:** `student/<name>-model`

This notebook contains two architectures as required by AT3:
- **Model 1** (Phase 2): Individual architecture
- **Model 2** (Phase 3): Refined architecture based on group discussion

> **Prerequisites:** The shared `data_preparation.ipynb` must have been run at least once  
> and its outputs uploaded to the shared Google Drive folder before running this notebook.

## 0. Install dependencies

In [1]:
!pip install -q torch torchvision nltk Pillow matplotlib

## 1. Mount Google Drive & load processed data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Shared Drive paths (must match data_preparation.ipynb) ───────────────────
DRIVE_FOLDER    = Path('/content/drive/MyDrive/AT3-DL-ImageCaptioning')
DRIVE_PROCESSED = DRIVE_FOLDER / 'processed'
DRIVE_IMAGES    = DRIVE_PROCESSED / 'images_224'
DRIVE_CHECKPTS  = DRIVE_FOLDER / 'checkpoints'

STUDENT_NAME = 'student_name'  # <-- CHANGE THIS
MY_CHECKPTS  = DRIVE_CHECKPTS / STUDENT_NAME
MY_CHECKPTS.mkdir(parents=True, exist_ok=True)

# Verify Drive artefacts exist
for p in [DRIVE_PROCESSED / 'vocab.pkl',
          DRIVE_PROCESSED / 'splits.json',
          DRIVE_PROCESSED / 'captions_clean.json']:
    status = 'OK' if p.exists() else 'MISSING — run data_preparation.ipynb first'
    print(f'[{status}] {p.name}')

n_imgs = len(list(DRIVE_IMAGES.glob('*.jpg')))
print(f'[OK] images_224/ — {n_imgs:,} images')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] vocab.pkl
[OK] splits.json
[OK] captions_clean.json
[OK] images_224/ — 0 images


In [3]:
import pickle, json, random
import numpy as np

with open(DRIVE_PROCESSED / 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

word2idx   = vocab['word2idx']
idx2word   = vocab['idx2word']
PAD_IDX    = vocab['PAD_IDX']
SOS_IDX    = vocab['SOS_IDX']
EOS_IDX    = vocab['EOS_IDX']
UNK_IDX    = vocab['UNK_IDX']
VOCAB_SIZE = len(word2idx)

with open(DRIVE_PROCESSED / 'splits.json') as f:
    splits = json.load(f)

with open(DRIVE_PROCESSED / 'captions_clean.json') as f:
    clean_data = json.load(f)

captions_clean = clean_data['captions']
id_to_filename = clean_data['id_to_filename']

print(f'Vocab size   : {VOCAB_SIZE:,}')
print(f'Train images : {len(splits["train"]):,}  Val: {len(splits["val"]):,}  Test: {len(splits["test"]):,}')

Vocab size   : 0
Train images : 6,200  Val: 775  Test: 775


## 2. Imports & device setup

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cpu


## 3. Dataset & DataLoader

In [5]:
class VizWizDataset(Dataset):
    """Loads 224x224 images from Drive and tokenised captions from memory."""

    def __init__(self, image_ids, captions, id_to_filename, img_dir, word2idx, transform=None):
        self.samples = []
        for img_id in image_ids:
            img_id_str = str(img_id)
            fname = id_to_filename.get(img_id_str)
            if fname is None:
                continue
            for tokens in captions.get(img_id_str, []):
                indices = ([SOS_IDX]
                           + [word2idx.get(t, UNK_IDX) for t in tokens]
                           + [EOS_IDX])
                self.samples.append((img_dir / fname, indices))

        self.transform = transform or T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, caption_indices = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        return self.transform(img), torch.tensor(caption_indices, dtype=torch.long)


def collate_fn(batch):
    images, captions = zip(*batch)
    images = torch.stack(images)
    lengths = [len(c) for c in captions]
    padded = torch.zeros(len(captions), max(lengths), dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :len(cap)] = cap
    return images, padded, torch.tensor(lengths)

In [6]:
BATCH_SIZE = 32

train_dataset = VizWizDataset(splits['train'], captions_clean, id_to_filename, DRIVE_IMAGES, word2idx)
val_dataset   = VizWizDataset(splits['val'],   captions_clean, id_to_filename, DRIVE_IMAGES, word2idx)
test_dataset  = VizWizDataset(splits['test'],  captions_clean, id_to_filename, DRIVE_IMAGES, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

Train batches: 969  Val: 122  Test: 122


---
# Model 1 — Phase 2 (Individual Architecture)

> **Describe your architecture rationale here before implementing.**  
> What encoder did you choose and why? What decoder and why?  
> What design decisions are specific to your approach?

In [7]:
class EncoderCNN(nn.Module):
    """Feature extractor using a pre-trained CNN."""

    def __init__(self, embed_size):
        super().__init__()
        # TODO: choose backbone (ResNet, EfficientNet, VGG, etc.) and adapt output
        pass

    def forward(self, images):
        # TODO: extract features, project to embed_size
        pass


class DecoderRNN(nn.Module):
    """Caption generator."""

    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super().__init__()
        # TODO: define embedding + RNN/LSTM/GRU + output projection
        pass

    def forward(self, features, captions):
        # TODO: teacher-forcing forward pass
        pass

    def generate(self, features, max_len=20):
        # TODO: greedy or beam-search inference
        pass

### Model 1 — Training

In [8]:
# Hyperparameters — document your choices here
EMBED_SIZE  = 256
HIDDEN_SIZE = 512
NUM_LAYERS  = 1
EPOCHS      = 10
LR          = 3e-4

encoder1 = EncoderCNN(EMBED_SIZE).to(DEVICE)
decoder1 = DecoderRNN(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE, NUM_LAYERS).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(list(encoder1.parameters()) + list(decoder1.parameters()), lr=LR)

# TODO: training loop
# Save checkpoints to MY_CHECKPTS / 'model1_epoch{e}.pth'

ValueError: optimizer got an empty parameter list

### Model 1 — Evaluation (BLEU-1 to BLEU-4)

In [ ]:
# TODO: generate captions for test split, compute BLEU-1, BLEU-2, BLEU-3, BLEU-4
# You can use nltk.translate.bleu_score or sacrebleu
pass

### Model 1 — Visual inspection

In [ ]:
# TODO: show 5-10 sample images with generated vs. reference captions
pass

---
# Group Discussion Summary

> *Fill in after the Phase 2 group meeting.*
>
> - What architectures did each teammate try?
> - What worked? What did not, and why?
> - What patterns emerged across experiments?
> - How did the discussion inform your Model 2 design?

---
# Model 2 — Phase 3 (Refined Architecture)

> **Describe changes made based on group discussion.**  
> Be explicit: what did you change, why, and what insight from Phase 2 drove that decision.

In [ ]:
# TODO: Model 2 architecture (encoder + decoder)
pass

### Model 2 — Training

In [ ]:
# TODO: training loop for Model 2
# Save checkpoints to MY_CHECKPTS / 'model2_epoch{e}.pth'
pass

### Model 2 — Evaluation & comparison vs. Model 1

In [ ]:
# TODO: BLEU scores for Model 2 + side-by-side comparison table with Model 1
pass